# Przykłady ocen jakości EyeQ

Losowe przykłady z ręcznymi ocenami jakości EyeQ (`metadata/eyeq/Label_EyeQ_train.csv`)
dla każdej kombinacji **klasa DR (0–4) × ocena (Good / Usable / Reject)** — po `N_PER_CELL` zdjęć.

Jedna figura na ocenę: wiersze = klasa DR, kolumny = kolejne przykłady.
Zmień `SEED`, żeby wylosować inne zdjęcia.

Zdjęcia pochodzą z cache'u (`scripts/build_cache.py`), ścieżka z `EYEPACS_CACHE`.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CACHE_DIR = Path(os.environ.get('EYEPACS_CACHE', '~/data/eyepacs/cache_768')).expanduser()
IMAGES_DIR = CACHE_DIR / 'images'
EYEQ_CSV = PROJECT_ROOT / 'metadata' / 'eyeq' / 'Label_EyeQ_train.csv'

N_PER_CELL = 5
SEED = 42
QUALITY_NAMES = {0: 'Good', 1: 'Usable', 2: 'Reject'}

eyeq = pd.read_csv(EYEQ_CSV, index_col=0)
eyeq['image'] = eyeq['image'].str.replace('.jpeg', '', regex=False)
eyeq['quality_name'] = eyeq['quality'].map(QUALITY_NAMES)

print('zdjęć z oceną EyeQ:', len(eyeq))
pd.crosstab(eyeq['DR_grade'], eyeq['quality_name'], margins=True)[['Good', 'Usable', 'Reject', 'All']]

In [ ]:
def load_thumb(image_name: str, size: int = 320) -> np.ndarray:
    image = cv2.imread(str(IMAGES_DIR / f'{image_name}.jpeg'))
    if image is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)


samples = eyeq.groupby(['quality', 'DR_grade']).sample(n=N_PER_CELL, random_state=SEED)
print('wylosowanych zdjęć:', len(samples))

In [ ]:
for quality, quality_name in QUALITY_NAMES.items():
    fig, axes = plt.subplots(5, N_PER_CELL, figsize=(N_PER_CELL * 3.2, 5 * 3.4))
    for grade in range(5):
        cell = samples[(samples['quality'] == quality) & (samples['DR_grade'] == grade)]
        for col in range(N_PER_CELL):
            ax = axes[grade, col]
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if col < len(cell):
                name = cell.iloc[col]['image']
                ax.imshow(load_thumb(name))
                ax.set_title(name, fontsize=9)
            if col == 0:
                ax.set_ylabel(f'klasa {grade}', fontsize=13)
    fig.suptitle(f'EyeQ: {quality_name}', fontsize=18)
    plt.tight_layout()
    plt.show()